In [ ]:
# A. EXTRACT

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Jika SparkSession belum tersedia
spark = SparkSession.builder \
    .appName("Tugas7_Pipeline_Analisis") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Path HDFS
base_path = "/user/mahasiswa/tugas7/raw"

# 1. Data cabang - JSON
df_cabang = spark.read \
    .option("multiline", "true") \
    .json(f"{base_path}/tugas7_cabang.json")

# 2. Data menu - CSV
df_menu = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(f"{base_path}/tugas7_menu.csv")

# 3. Data transaksi - CSV
df_transaksi = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(f"{base_path}/tugas7_transaksi.csv")

# Tampilkan schema
print("=== Schema Cabang ===")
df_cabang.printSchema()

print("=== Schema Menu ===")
df_menu.printSchema()

print("=== Schema Transaksi ===")
df_transaksi.printSchema()

# Tampilkan beberapa data
print("=== Data Cabang ===")
df_cabang.show(5, truncate=False)

print("=== Data Menu ===")
df_menu.show(5, truncate=False)

print("=== Data Transaksi ===")
df_transaksi.show(5, truncate=False)


In [ ]:
# B. TRANSFORM — Join + Kolom Turunan

df_gabungan = df_transaksi \
    .join(
        df_menu,
        df_transaksi.menu_id == df_menu.menu_id,
        "inner"
    ) \
    .join(
        df_cabang,
        df_transaksi.cabang_id == df_cabang.cabang_id,
        "inner"
    ) \
    .withColumn(
        "total_penjualan",
        col("qty") * col("harga")
    )

# Pilih kolom yang relevan
df_gabungan = df_gabungan.select(
    df_transaksi.trx_id,
    df_transaksi.cabang_id,
    df_cabang.nama_cabang,
    df_cabang.kota,
    df_cabang.kepala_barista,
    df_transaksi.menu_id,
    df_menu.nama_menu,
    df_menu.kategori_menu,
    df_menu.harga,
    df_transaksi.qty,
    col("total_penjualan")
)

print("=== Schema Data Gabungan ===")
df_gabungan.printSchema()

print("=== Contoh Data Gabungan ===")
df_gabungan.show(10, truncate=False)


In [ ]:
# C. TRANSFORM — Window Function, Top-2 Menu per Cabang

# Agregasi penjualan per cabang dan menu
df_penjualan_menu = df_gabungan.groupBy(
    "cabang_id",
    "nama_cabang",
    "kota",
    "menu_id",
    "nama_menu",
    "kategori_menu"
).agg(
    sum("qty").alias("total_qty"),
    sum("total_penjualan").alias("total_penjualan")
)

# Window: ranking berdasarkan total penjualan per cabang
window_cabang = Window \
    .partitionBy("cabang_id") \
    .orderBy(col("total_penjualan").desc())

# Berikan nomor ranking
df_top2 = df_penjualan_menu \
    .withColumn(
        "rn",
        row_number().over(window_cabang)
    ) \
    .filter(col("rn") <= 2) \
    .orderBy("cabang_id", "rn")

print("=== TOP-2 MENU TERLARIS SETIAP CABANG ===")

df_top2.select(
    "cabang_id",
    "nama_cabang",
    "menu_id",
    "nama_menu",
    "kategori_menu",
    "total_qty",
    "total_penjualan",
    "rn"
).show(20, truncate=False)


In [ ]:
# D. TRANSFORM — Spark SQL

spark.sql("""
    SELECT
        kepala_barista,
        SUM(total_penjualan) AS total_penjualan
    FROM transaksi_gabungan
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
""").select(
    "kepala_barista",
    format_number("total_penjualan", 0).alias("total_penjualan")
).show(truncate=False)


In [ ]:
# E. LOAD — Simpan sebagai Parquet dengan Partition

output_path = "/user/mahasiswa/tugas7/processed/laporan"

df_gabungan.write \
    .mode("overwrite") \
    .partitionBy("kategori_menu") \
    .parquet(output_path)

print("Data berhasil disimpan ke:")
print(output_path)

# verifikasi bahwa Parquet bisa dibaca kembali
df_laporan = spark.read.parquet(output_path)

print("=== Schema Parquet ===")
df_laporan.printSchema()

print("=== Data Parquet ===")
df_laporan.show(10, truncate=False)

print("Jumlah data:", df_laporan.count())

# untuk melihat struktur partisinya
print("Data tersimpan dalam format Parquet dengan partition berdasarkan kategori_menu.")

# Kalau ingin mengecek lewat HDFS dari notebook
import subprocess

result = subprocess.run(
    ["hdfs", "dfs", "-ls", output_path],
    capture_output=True,
    text=True
)

print(result.stdout)


In [ ]:
# F. Ringkasan Eksekutif & Rekomendasi

# 1. Tabel ringkasan per cabang
ringkasan_cabang = df_gabungan.groupBy(
    "cabang_id",
    "nama_cabang",
    "kota"
).agg(
    sum("total_penjualan").alias("total_penjualan"),
    countDistinct("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(
    col("total_penjualan").desc()
)

print("=== RINGKASAN PER CABANG ===")

ringkasan_cabang.select(
    "cabang_id",
    "nama_cabang",
    "kota",
    format_number("total_penjualan", 0).alias("total_penjualan"),
    "jumlah_transaksi",
    format_number(
        "rata_rata_nilai_transaksi", 2
    ).alias("rata_rata_nilai_transaksi")
).show(truncate=False)

# 2. Buat tabel rekomendasi berdasarkan hasil analisis

cabang_tertinggi = ringkasan_cabang.orderBy(
    col("total_penjualan").desc()
).first()

cabang_terendah = ringkasan_cabang.orderBy(
    col("total_penjualan").asc()
).first()

print("=== CABANG TOTAL PENJUALAN TERTINGGI ===")
print(cabang_tertinggi)

print("\n=== CABANG TOTAL PENJUALAN TERENDAH ===")
print(cabang_terendah)

# Markdown F — Ringkasan Eksekutif
# Ringkasan Eksekutif & Rekomendasi

